# 🌡️ Analisis & Prediksi Cuaca IoT — Pengamatan 3
**Mata Kuliah:** Kecerdasan Buatan  
**Dataset Primer:** `sensor_data_3.csv` — sensor IoT outdoor (suhu, kelembapan, cahaya LDR)  
**Dataset Eksternal:** Open-Meteo API + Visual Crossing (Banyumanik, Semarang)  
**Lokasi sensor:** Teras — terpapar sinar matahari langsung  
**Periode:** 13 – 22 April 2026 (WIB, GMT+7)  

---

## 🗺️ Alur Analisis
| # | Tahap | Deskripsi |
|---|-------|-----------|
| 1 | Import Library | Setup environment |
| 2 | Load & Inspect Data | Muat semua sumber data |
| 3 | Preprocessing | Konversi timezone, koreksi LDR |
| 4 | Audit Kualitas Data | Identifikasi gap besar & kecil |
| 5 | **Optimasi Imputasi** | Gap besar → data eksternal, gap kecil → interpolasi |
| 6 | Feature Engineering | Fitur temporal, lag, siklus, data cuaca eksternal |
| 7 | EDA | Univariat, bivariat, multivariat, time-series story |
| 8 | Deteksi Outlier | IQR + Z-Score |
| 9 | **Prediksi Suhu** | Linear Regression, Random Forest, ARIMA Walk-Forward |
| 10 | Kesimpulan | Temuan utama & rekomendasi |

## 1. Import Library

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.colors as LinearSegmentedColormap
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX

# color palette
PALETTE = {
    'suhu'    : '#E53935',
    'lembap'  : '#1E88E5',
    'cahaya'  : '#F9A825',
    'ext'     : '#43A047',
    'gap_big' : '#EF9A9A',
    'gap_sm'  : '#FFF9C4',
    'night'   : '#E8EAF6',
    'accent'  : '#6C63FF',
    'dark'    : '#1A1A2E',
}

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi'      : 130,
    'font.family'     : 'DejaVu Sans',
    'font.size'       : 11,
    'axes.titlesize'  : 13,
    'axes.titleweight': 'bold',
    'axes.labelsize'  : 11,
    'axes.spines.top' : False,
    'axes.spines.right': False,
    'axes.grid'       : True,
    'grid.alpha'      : 0.3,
    'legend.framealpha': 0.85,
    'figure.facecolor': 'white',
    'axes.facecolor'  : '#FAFAFA',
})

print('Sucessfully import all libraries')

Sucessfully import all libraries


## 2. Load & Inspect Data

dengan tiga sumber data: sensor IoT, Open-Meteo (reanalysis model) and Visual Crossing (observasi stasiun terdekat)

In [2]:
# sensor IoT
df_raw = pd.read_csv('sensor_data_3.csv')
print(f'Sensor  → {df_raw.shape[0]} baris × {df_raw.shape[1]} kolom')
display(df_raw.head(3))

# open meteo (hourly, GMT +7)
df_om_raw = pd.read_csv('open-meteo.csv', skiprows=3)
df_om_raw.rename(columns={
    'time'                      : 'timestamp',
    'temperature_2m (°C)'       : 'ext_suhu_om',
    'relative_humidity_2m (%)'  : 'ext_lembap_om',
    'cloud_cover (%)'           : 'ext_awan_om',
    'direct_radiation (W/m²)'   : 'ext_rad_dir_om',
    'diffuse_radiation (W/m²)'  : 'ext_rad_dif_om',
}, inplace=True)
df_om_raw['timestamp'] = pd.to_datetime(df_om_raw['timestamp']).dt.tz_localize('Asia/Jakarta')
df_om_raw.set_index('timestamp', inplace=True)
df_om_raw['ext_rad_total_om'] = df_om_raw['ext_rad_dir_om'] + df_om_raw['ext_rad_dif_om']
print(f'Open-Meteo → {len(df_om_raw)} baris  |  {df_om_raw.index.min().date()} → {df_om_raw.index.max().date()}')

# visual crossing
df_vc_raw = pd.read_csv('visual-crossing.csv')
df_vc_raw['timestamp'] = pd.to_datetime(df_vc_raw['datetime']).dt.tz_localize('Asia/Jakarta')
df_vc_raw.set_index('timestamp', inplace=True)
df_vc_raw.rename(columns={
    'temp'           : 'ext_suhu_vc',
    'humidity'       : 'ext_lembap_vc',
    'solarradiation' : 'ext_solar_vc',
    'cloudcover'     : 'ext_awan_vc',
    'windspeed'      : 'ext_angin_vc',
    'sealevelpressure': 'ext_pressure_vc',
    'precipprob'     : 'ext_precipprob_vc',
}, inplace=True)
print(f'Visual Crossing → {len(df_vc_raw)} baris  |  {df_vc_raw.index.min().date()} → {df_vc_raw.index.max().date()}')

print()
print('successfully load all datas')

Sensor  → 2082 baris × 6 kolom


,id,created_at,suhu,kelembapan,cahaya,kondisi
0,18,2026-04-13 02:52:00.210695+00,29.8,71.5,711,TERANG
1,19,2026-04-13 02:57:00.190764+00,29.8,70.9,400,TERANG
2,20,2026-04-13 03:03:00.467898+00,30.0,68.5,389,TERANG


Open-Meteo → 240 baris  |  2026-04-13 → 2026-04-22
Visual Crossing → 240 baris  |  2026-04-13 → 2026-04-22

successfully load all datas


## 3. Preprocessing
### 3.1 Konversi Timezone UTC → WIB (GMT+7)
Data sensor tersimpan dalam **UTC** di backend Supabase. Sensor berada di Semarang (WIB = GMT+7), sehingga tanpa konversi pola harian akan bergeser 7 jam.

### 3.2 Koreksi Nilai Sensor Cahaya (LDR)
LDR bekerja **terbalik** dari intuisi:
- `cahaya_raw` rendah → resistansi rendah → **TERANG** (sinar matahari)  
- `cahaya_raw` tinggi → resistansi tinggi → **GELAP**

Koreksi: `cahaya = 4095 - cahaya_raw`  
Setelah koreksi: nilai tinggi = terang — konsisten secara fisik ✅

In [4]:
df = df_raw.copy()

# parse and convert utc -> wib
df['timestamp'] = pd.to_datetime(df['created_at'], utc=True)
df['timestamp'] = df['timestamp'].dt.tz_convert('Asia/Jakarta')
df = df.sort_values('timestamp').reset_index(drop=True)
df = df.set_index('timestamp')
df.index.name = 'timestamp'

df['suhu']       = pd.to_numeric(df['suhu'], errors='coerce')
df['kelembapan'] = pd.to_numeric(df['kelembapan'], errors='coerce')
df['cahaya_raw'] = pd.to_numeric(df['cahaya'], errors='coerce')

# correction ldr value
df['cahaya'] = 4095 - df['cahaya_raw']

print(f'✅ Preprocessing done')
print(f'   Rentang waktu (WIB): {df.index.min().strftime("%d %b %Y %H:%M")} → {df.index.max().strftime("%d %b %Y %H:%M")}')
print(f'   Durasi total        : {df.index.max() - df.index.min()}')

# verif koreksi cahaya
tmp = df.copy()
tmp['jam_h'] = tmp.index.hour
siang = tmp[tmp['jam_h'].between(10, 15)]
malam = tmp[(tmp['jam_h'] >= 21) | (tmp['jam_h'] <= 4)]
print()

print('✅ Koreksi LDR berhasil')
print(f'  Siang (10-15 WIB): cahaya mean = {siang["cahaya"].mean():.0f}  ← harusnya TINGGI')
print(f'  Malam (21-04 WIB): cahaya mean = {malam["cahaya"].mean():.0f}  ← harusnya RENDAH')
display(df[['suhu','kelembapan','cahaya_raw','cahaya','kondisi']].head(5))

✅ Preprocessing done
   Rentang waktu (WIB): 13 Apr 2026 09:52 → 22 Apr 2026 13:08
   Durasi total        : 9 days 03:16:24.106716

✅ Koreksi LDR berhasil
  Siang (10-15 WIB): cahaya mean = 3923  ← harusnya TINGGI
  Malam (21-04 WIB): cahaya mean = 0  ← harusnya RENDAH


,suhu,kelembapan,cahaya_raw,cahaya,kondisi
timestamp,,,,,
2026-04-13 09:52:00.210695+07:00,29.8,71.5,711,3384,TERANG
2026-04-13 09:57:00.190764+07:00,29.8,70.9,400,3695,TERANG
2026-04-13 10:03:00.467898+07:00,30.0,68.5,389,3706,TERANG
2026-04-13 10:08:00.503225+07:00,29.8,69.9,260,3835,TERANG
2026-04-13 10:13:00.613372+07:00,30.4,68.1,247,3848,TERANG


## 4. Audit Kualitas Data
Identifikasi tiga kategori masalah:
1. **Gap Besar** (> 60 menit) -> sensor tidak mengirim data; akan diisi dengan data eksternal  
2. **Gap Kecil** (10 – 60 menit) -> jitter jaringan ringan; akan diisi interpolasi  
3. **Jitter** (5 – 10 menit) -> variasi normal interval pengiriman

In [10]:
df_temp = df.reset_index()
df_temp['gap_menit'] = df_temp['timestamp'].diff().dt.total_seconds() / 60

q = df_temp['gap_menit'].dropna()
print('=== DISTRIBUSI INTERVAL PENGIRIMAN ===')
print(f'  Median interval   : {q.median():.0f} menit')
print(f'  95th percentile   : {q.quantile(0.95):.1f} menit')
print(f'  Gap > 10 menit    : {(q>10).sum()} kejadian')
print(f'  Gap > 60 menit    : {(q>60).sum()} kejadian')

# gap classification
big_gaps_df = df_temp[df_temp['gap_menit'] > 60][['timestamp', 'gap_menit']].copy()
small_gaps_df = df_temp[(df_temp['gap_menit'] > 10) & (df_temp['gap_menit'] <= 60)][['timestamp', 'gap_menit']].copy()

big_gaps_df['start'] = [df_temp.loc[i-1,'timestamp'] if i>0 else None for i in big_gaps_df.index]
small_gaps_df['start'] = [df_temp.loc[i-1,'timestamp'] if i>0 else None for i in small_gaps_df.index]
big_gaps_df.rename(columns={'timestamp':'end'}, inplace=True)
small_gaps_df.rename(columns={'timestamp':'end'}, inplace=True)

print(f'\n=== Gap Besar (>60 menit): {len(big_gaps_df)} kejadian ===')
for _, row in big_gaps_df.iterrows():
  print(f'{row["start"].strftime("%d/%m %H:%M")} → {row["end"].strftime("%d/%m %H:%M")}  ({row["gap_menit"]/60:.1f} jam)')

print(f'\n=== Gap Kecil (10-60 menit): {len(small_gaps_df)} kejadian ===')
# print(small_gaps_df[['start', 'end', 'gap_menit']].to_string(index=False))

=== DISTRIBUSI INTERVAL PENGIRIMAN ===
  Median interval   : 5 menit
  95th percentile   : 10.1 menit
  Gap > 10 menit    : 139 kejadian
  Gap > 60 menit    : 2 kejadian

=== Gap Besar (>60 menit): 2 kejadian ===
13/04 21:43 → 14/04 13:09  (15.4 jam)
18/04 19:46 → 19/04 08:29  (12.7 jam)

=== Gap Kecil (10-60 menit): 137 kejadian ===
